# CFM 界面刚度各向异性：独立模拟组合分布

目标：对每组三个界面方向，使用每个方向的三次独立模拟结果形成 27 种组合，计算界面刚度各向异性参数 $\epsilon_1$ 和 $\epsilon_2$ 的平均值与标准差。

注意：不使用三个 110 方向同时构成的一组，因为它们线性相关。

In [ ]:
from pathlib import Path
import os


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in (start, *start.parents):
        if (path / "interface_analyzer" / "reproducibility").exists():
            return path
    raise RuntimeError("Could not find repository root containing interface_analyzer/reproducibility")


PROJECT_ROOT = find_repo_root()
REPRO_DIR = PROJECT_ROOT / "interface_analyzer" / "reproducibility"
DATASET_DIR = REPRO_DIR / "dataset"
LOCAL_OUTPUT_DIR = REPRO_DIR / "_local_outputs"
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Full manuscript-scale post-processing files are intentionally not bundled.
# Set INTERFACE_ANALYZER_DATA to the directory containing those generated files.
FULL_DATA_ROOT = Path(os.environ.get("INTERFACE_ANALYZER_DATA", LOCAL_OUTPUT_DIR)).expanduser()
FULL_DATA_ROOT.mkdir(parents=True, exist_ok=True)

# For quick local CFG tests this defaults to the bundled sample dataset.
CFG_DIR = Path(os.environ.get("INTERFACE_ANALYZER_CFG_DIR", DATASET_DIR)).expanduser()

print("Project root:", PROJECT_ROOT)
print("Bundled CFG dataset:", DATASET_DIR)
print("Analysis data root:", FULL_DATA_ROOT)
print("CFG input dir:", CFG_DIR)


## 1. 输入数据

In [ ]:
import itertools
import os
from collections import OrderedDict

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import matplotlib.pyplot as plt
import numpy as np


# 界面刚度数据：每个方向三次独立模拟，单位沿用原 notebook 的 E-20。
STIFFNESS_SAMPLES_RAW = {
    "100[010]":  [86.3, 85.3, 83.7],
    "110[001]":  [80.2, 79.9, 78.9],
    "110[1-10]": [121.7, 124.1, 122.7],
    "110[1-12]": [95.5, 93.9, 93.9],
    "111[1-21]": [110.2, 110.7, 112.7],
}

UNIT_SCALE = 1e-20
STIFFNESS_SAMPLES = {
    name: np.array(values, dtype=float) * UNIT_SCALE
    for name, values in STIFFNESS_SAMPLES_RAW.items()
}

# beta = gamma0 * (c0 + c1*epsilon1 + c2*epsilon2)
STIFFNESS_MODELS = {
    "100[010]":  (1.0, -18/5,   -80/7),
    "110[001]":  (1.0, -21/10,  365/14),
    "110[1-10]": (1.0,  39/10,  155/14),
    "110[1-12]": (1.0, -1/10,   295/14),
    "111[1-21]": (1.0,  12/5,  -1280/63),
}

LINEAR_DEPENDENT_110 = {"110[001]", "110[1-10]", "110[1-12]"}

## 2. 计算函数

In [ ]:
def solve_from_combo(stiffness_dict, combo, models=STIFFNESS_MODELS):
    """由三个方向的刚度求 gamma0、epsilon1、epsilon2。"""
    A = np.array([models[name] for name in combo], dtype=float)
    b = np.array([stiffness_dict[name] for name in combo], dtype=float)

    # x = [gamma0, gamma0*epsilon1, gamma0*epsilon2]
    x1, x2, x3 = np.linalg.solve(A, b)
    gamma0 = x1
    epsilon1 = x2 / x1
    epsilon2 = x3 / x1

    return gamma0, epsilon1, epsilon2, np.linalg.cond(A)


def valid_direction_combos(samples=STIFFNESS_SAMPLES):
    """列出所有可用的三个方向组合，并去掉三个 110 的线性相关组合。"""
    names = list(samples)
    combos = []

    for combo in itertools.combinations(names, 3):
        if set(combo) == LINEAR_DEPENDENT_110:
            continue
        combos.append(combo)

    return combos


def solve_all_replicate_combinations(samples=STIFFNESS_SAMPLES):
    """对每个方向组合计算 3^3=27 种独立模拟组合。"""
    rows = []

    for combo in valid_direction_combos(samples):
        for sample_indices in itertools.product(range(3), repeat=3):
            stiffness_dict = {
                name: samples[name][sample_idx]
                for name, sample_idx in zip(combo, sample_indices)
            }
            gamma0, epsilon1, epsilon2, cond_A = solve_from_combo(stiffness_dict, combo)

            rows.append({
                "direction_combo": " + ".join(combo),
                "sample_indices": sample_indices,
                "gamma0": gamma0,
                "epsilon1": epsilon1,
                "epsilon2": epsilon2,
                "minus_epsilon2": -epsilon2,
                "cond_A": cond_A,
            })

    return rows


def summarize_by_direction_combo(results):
    """每个方向组合的 27 个结果给出均值和样本标准差。"""
    grouped = OrderedDict()

    for row in results:
        grouped.setdefault(row["direction_combo"], []).append(row)

    summary = []
    for direction_combo, rows in grouped.items():
        summary.append({
            "direction_combo": direction_combo,
            "n": len(rows),
            "gamma0_mean": np.mean([row["gamma0"] for row in rows]),
            "gamma0_std": np.std([row["gamma0"] for row in rows], ddof=1),
            "epsilon1_mean": np.mean([row["epsilon1"] for row in rows]),
            "epsilon1_std": np.std([row["epsilon1"] for row in rows], ddof=1),
            "epsilon2_mean": np.mean([row["epsilon2"] for row in rows]),
            "epsilon2_std": np.std([row["epsilon2"] for row in rows], ddof=1),
            "minus_epsilon2_mean": np.mean([row["minus_epsilon2"] for row in rows]),
            "minus_epsilon2_std": np.std([row["minus_epsilon2"] for row in rows], ddof=1),
            "cond_A": rows[0]["cond_A"],
        })

    return summary


def print_table(rows, columns):
    """打印一个紧凑表格，避免额外依赖 pandas。"""
    def format_value(value):
        if isinstance(value, (float, np.floating)):
            return f"{value:.10e}"
        return str(value)

    formatted_rows = [
        [format_value(row[column]) for column in columns]
        for row in rows
    ]
    widths = [
        max(len(column), *(len(row[i]) for row in formatted_rows))
        for i, column in enumerate(columns)
    ]

    header = "  ".join(column.ljust(width) for column, width in zip(columns, widths))
    print(header)
    print("  ".join("-" * width for width in widths))
    for row in formatted_rows:
        print("  ".join(value.ljust(width) for value, width in zip(row, widths)))

## 3. 计算不同组合的分布

In [ ]:
results = solve_all_replicate_combinations()
summary = summarize_by_direction_combo(results)

print(f"方向组合数: {len(summary)}")
print(f"独立模拟组合总数: {len(results)}")

print_table(
    summary,
    [
        "direction_combo",
        "n",
        "epsilon1_mean",
        "epsilon1_std",
        "epsilon2_mean",
        "epsilon2_std",
        "cond_A",
    ],
)

## 4. 总体统计

In [ ]:
overall_stats = []
for quantity in ["gamma0", "epsilon1", "epsilon2", "minus_epsilon2"]:
    values = np.array([row[quantity] for row in results], dtype=float)
    label = "-epsilon2" if quantity == "minus_epsilon2" else quantity
    overall_stats.append({
        "quantity": label,
        "mean": np.mean(values),
        "std": np.std(values, ddof=1),
    })

print_table(overall_stats, ["quantity", "mean", "std"])

## 5. $\epsilon_1$ 与 $-\epsilon_2$ 散点图

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

colors = plt.cm.tab10(np.linspace(0, 1, len(summary)))

for color, row in zip(colors, summary):
    combo_points = [
        point for point in results
        if point["direction_combo"] == row["direction_combo"]
    ]

    ax.scatter(
        [point["minus_epsilon2"] for point in combo_points],
        [point["epsilon1"] for point in combo_points],
        s=18,
        alpha=0.22,
        color=color,
    )

    ax.errorbar(
        row["minus_epsilon2_mean"],
        row["epsilon1_mean"],
        xerr=row["minus_epsilon2_std"],
        yerr=row["epsilon1_std"],
        fmt="o",
        capsize=4,
        markersize=6,
        color=color,
        label=row["direction_combo"],
    )

x_min, x_max = ax.get_xlim()
x_line = np.linspace(x_min, x_max, 200)
ax.plot(
    x_line,
    (20 / 3) * x_line,
    "k--",
    linewidth=1.5,
    label=r"$\epsilon_1=-\frac{20}{3}\epsilon_2$",
)

ax.set_xlabel(r"$-\epsilon_2$")
ax.set_ylabel(r"$\epsilon_1$")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8, loc="best")
fig.tight_layout()

## 6. 输出 txt 文件

输出三个文件，便于在 Origin 中画图：

- `epsilon_replicate_results.txt`：所有 243 个独立模拟组合结果。
- `epsilon_summary_by_direction_combo.txt`：每个方向组合的均值与标准差。
- `epsilon_origin_errorbar_data.txt`：Origin 画散点误差棒最常用的列。

In [ ]:
def write_table_txt(path, rows, columns):
    """写出制表符分隔的 txt 文件，方便 Origin 直接导入。"""
    with open(path, "w", encoding="utf-8") as f:
        f.write("\t".join(columns) + "\n")
        for row in rows:
            values = []
            for column in columns:
                value = row[column]
                if isinstance(value, tuple):
                    value = ",".join(str(item + 1) for item in value)
                elif isinstance(value, (float, np.floating)):
                    value = f"{value:.12e}"
                values.append(str(value))
            f.write("\t".join(values) + "\n")


replicate_columns = [
    "direction_combo",
    "sample_indices",
    "gamma0",
    "epsilon1",
    "epsilon2",
    "minus_epsilon2",
    "cond_A",
]

summary_columns = [
    "direction_combo",
    "n",
    "gamma0_mean",
    "gamma0_std",
    "epsilon1_mean",
    "epsilon1_std",
    "epsilon2_mean",
    "epsilon2_std",
    "minus_epsilon2_mean",
    "minus_epsilon2_std",
    "cond_A",
]

origin_rows = []
for row in summary:
    origin_rows.append({
        "direction_combo": row["direction_combo"],
        "x_minus_epsilon2_mean": row["minus_epsilon2_mean"],
        "xerr_minus_epsilon2_std": row["minus_epsilon2_std"],
        "y_epsilon1_mean": row["epsilon1_mean"],
        "yerr_epsilon1_std": row["epsilon1_std"],
    })

origin_columns = [
    "direction_combo",
    "x_minus_epsilon2_mean",
    "xerr_minus_epsilon2_std",
    "y_epsilon1_mean",
    "yerr_epsilon1_std",
]

write_table_txt("epsilon_replicate_results.txt", results, replicate_columns)
write_table_txt("epsilon_summary_by_direction_combo.txt", summary, summary_columns)
write_table_txt("epsilon_origin_errorbar_data.txt", origin_rows, origin_columns)

print("已输出:")
print("epsilon_replicate_results.txt")
print("epsilon_summary_by_direction_combo.txt")
print("epsilon_origin_errorbar_data.txt")